# Fakeddit multimodal methodology audit — v2

This notebook adds the methodological checks that were missing from the original project while preserving the original code, data, checkpoints, and results. It reads from the sibling `multimodal-fake-news-detection` folder and writes only to this `v2` folder.

The notebook covers:

1. binary-label semantics and internal consistency;
2. class-dependent image-selection bias;
3. a fairer validation-tuned and calibrated RoBERTa comparison;
4. source, domain, and time subgroup performance plus a source-only shortcut baseline;
5. perceptual-image duplicate screening;
6. source-disjoint and chronological manifests for stronger future retraining;
7. explainability coverage and reporting consistency checks.

The final holdout is used only for locked evaluation of predeclared controls. No original model or result is overwritten.

In [ ]:
from pathlib import Path
import json, hashlib, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.fft import dctn
from scipy.stats import chi2_contingency
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    log_loss, confusion_matrix
)
from sklearn.model_selection import train_test_split

# Open the notebook from v2 or the submission repository root.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "v2":
    REPO_ROOT = REPO_ROOT.parent
ORIGINAL_ROOT = REPO_ROOT / "multimodal-fake-news-detection"
V2_ROOT = REPO_ROOT / "v2"
OUTPUT_ROOT = V2_ROOT / "outputs"
MANIFEST_ROOT = V2_ROOT / "manifests"
RAW_ROOT = ORIGINAL_ROOT / "data" / "raw" / "fakeddit"
PROCESSED_ROOT = ORIGINAL_ROOT / "data" / "processed" / "fakeddit"
RESULTS_ROOT = ORIGINAL_ROOT / "results"
SEED = 2026

assert ORIGINAL_ROOT.exists(), ORIGINAL_ROOT
assert V2_ROOT != ORIGINAL_ROOT and ORIGINAL_ROOT not in V2_ROOT.parents
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

def save_json(name, obj):
    path = OUTPUT_ROOT / name
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    return path

def raw_path(split):
    name = {"train":"multimodal_train.tsv", "validation":"multimodal_validate.tsv", "test":"multimodal_test_public.tsv"}[split]
    return RAW_ROOT / "multimodal_only_samples" / name

def read_raw(split, columns=None):
    return pd.read_csv(raw_path(split), sep="\t", usecols=columns, low_memory=False)

def metrics(y, p, threshold=0.5):
    y = np.asarray(y, dtype=int); p = np.clip(np.asarray(p, dtype=float), 1e-7, 1-1e-7)
    pred = (p >= threshold).astype(int)
    return {
        "n": int(len(y)), "threshold": float(threshold),
        "accuracy": float(accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "roc_auc": float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan,
        "pr_auc": float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else np.nan,
        "brier": float(brier_score_loss(y, p)),
        "log_loss": float(log_loss(y, p, labels=[0,1])),
        "confusion_matrix": confusion_matrix(y, pred, labels=[0,1]).tolist(),
    }

print("Original (read-only):", ORIGINAL_ROOT)
print("v2 outputs:", V2_ROOT)

## 1. Verify label semantics

Fakeddit's six-way label `0` is the true-content category; the other five categories are forms of false or misleading content. The binary encoding is therefore checked programmatically: binary class `1` must correspond exactly to six-way class `0`, and binary class `0` to six-way classes `1–5`.

In [ ]:
label_rows = []
label_tables = {}
for split in ["train", "validation", "test"]:
    d = read_raw(split, ["id", "2_way_label", "3_way_label", "6_way_label"])
    tab = pd.crosstab(d["2_way_label"], d["6_way_label"])
    label_tables[split] = tab
    consistent = ((d["2_way_label"] == 1) == (d["6_way_label"] == 0))
    label_rows.append({
        "split": split, "n": len(d), "consistent_n": int(consistent.sum()),
        "inconsistent_n": int((~consistent).sum()), "consistent_pct": float(100*consistent.mean())
    })
    display(tab)

label_audit = pd.DataFrame(label_rows)
label_audit.to_csv(OUTPUT_ROOT / "label_semantics_audit.csv", index=False)
assert label_audit["inconsistent_n"].sum() == 0
semantics = {
    "binary_class_1": "true/real content",
    "binary_class_0": "fake or misleading content (six-way classes 1-5)",
    "verification": label_rows,
    "limitation": "This verifies dataset encoding, not the factual correctness of every distant-supervision label."
}
save_json("label_semantics.json", semantics)
display(label_audit)

## 2. Quantify selection bias caused by image availability

Equal sample IDs make comparisons between modalities fair within the retained subset. They do not make that subset representative. This section measures retention by label, year, domain, and subreddit and tests whether retention is independent of the binary label.

In [ ]:
status_parts = []
for split in ["train", "validation", "test"]:
    p = PROCESSED_ROOT / "development_clean" / f"{split}_with_image_status.parquet"
    s = pd.read_parquet(p)
    meta = read_raw(split, ["id", "subreddit", "domain", "created_utc", "2_way_label"])
    s["id"] = s["id"].astype(str); meta["id"] = meta["id"].astype(str)
    s = s.merge(meta, on="id", how="left", validate="one_to_one")
    s["retained"] = s["image_usable"].fillna(False).astype(bool)
    s["year"] = pd.to_datetime(s["created_utc"], unit="s", errors="coerce", utc=True).dt.year
    status_parts.append(s)
status = pd.concat(status_parts, ignore_index=True)

retention = (status.groupby(["split", "label"])["retained"]
             .agg(total="size", retained="sum", retention_rate="mean").reset_index())
retention["removed"] = retention["total"] - retention["retained"]
retention.to_csv(OUTPUT_ROOT / "selection_retention_by_label.csv", index=False)

contingency = pd.crosstab(status["label"], status["retained"])
chi2, p_value, _, _ = chi2_contingency(contingency)
n = contingency.to_numpy().sum()
cramers_v = math.sqrt(chi2 / n)
bias_summary = {
    "n": int(n), "chi_square": float(chi2), "p_value": float(p_value),
    "cramers_v": float(cramers_v),
    "interpretation": "Large dependence means complete-case filtering changes the target population. Macro-F1 does not remove this selection bias."
}
save_json("selection_bias_summary.json", bias_summary)

for group in ["subreddit", "domain", "year"]:
    g = (status.groupby(group, dropna=False)["retained"]
         .agg(total="size", retained="sum", retention_rate="mean").reset_index())
    g["removed"] = g["total"] - g["retained"]
    g.sort_values(["total", "retention_rate"], ascending=[False, True]).to_csv(
        OUTPUT_ROOT / f"selection_retention_by_{group}.csv", index=False)

display(retention)
print(bias_summary)

fig, ax = plt.subplots(figsize=(7,4))
for split, grp in retention.groupby("split"):
    ax.plot(grp["label"].astype(str), grp["retention_rate"], marker="o", label=split)
ax.set(xlabel="Binary label (0=fake/misleading, 1=true)", ylabel="Image retention rate", ylim=(0,1),
       title="Image filtering is strongly class-dependent")
ax.legend(); fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "selection_retention_by_label.png", dpi=180)
plt.show()

## 3. Fair text-only threshold and calibration controls

The original fusion weight was selected on validation macro-F1 while RoBERTa used the default 0.5 threshold. This section gives RoBERTa the same opportunity to tune its decision threshold. It also fits Platt calibration on one stratified half of validation and selects its threshold on the other half, then evaluates each predeclared variant once on the frozen holdout.

In [ ]:
val = pd.read_csv(RESULTS_ROOT / "metrics" / "baseline_probabilities" / "roberta_validation.csv")
final = pd.read_csv(RESULTS_ROOT / "metrics" / "final_evaluation" / "final_predictions.csv")
val = val.rename(columns={"probability_class_1":"p_text"})

threshold_grid = np.linspace(0.05, 0.95, 181)
threshold_scores = pd.DataFrame({
    "threshold": threshold_grid,
    "validation_macro_f1": [metrics(val.label, val.p_text, t)["macro_f1"] for t in threshold_grid]
})
best_t = float(threshold_scores.sort_values(["validation_macro_f1", "threshold"], ascending=[False, True]).iloc[0].threshold)
threshold_scores.to_csv(OUTPUT_ROOT / "roberta_validation_threshold_search.csv", index=False)

idx_cal, idx_tune = train_test_split(np.arange(len(val)), test_size=0.5, random_state=SEED, stratify=val.label)
def logit(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1-1e-6)
    return np.log(p/(1-p)).reshape(-1,1)
platt = LogisticRegression(C=1e6, solver="lbfgs", random_state=SEED)
platt.fit(logit(val.p_text.iloc[idx_cal]), val.label.iloc[idx_cal])
p_tune = platt.predict_proba(logit(val.p_text.iloc[idx_tune]))[:,1]
platt_grid = pd.DataFrame({
    "threshold": threshold_grid,
    "tuning_half_macro_f1": [metrics(val.label.iloc[idx_tune], p_tune, t)["macro_f1"] for t in threshold_grid]
})
platt_t = float(platt_grid.sort_values(["tuning_half_macro_f1", "threshold"], ascending=[False, True]).iloc[0].threshold)
platt_grid.to_csv(OUTPUT_ROOT / "roberta_platt_threshold_search.csv", index=False)
p_final_platt = platt.predict_proba(logit(final.probability_roberta))[:,1]

control_results = {
    "roberta_default_0.5": metrics(final.label, final.probability_roberta, 0.5),
    "roberta_validation_tuned_threshold": metrics(final.label, final.probability_roberta, best_t),
    "roberta_platt_default_0.5": metrics(final.label, p_final_platt, 0.5),
    "roberta_platt_tuned_threshold": metrics(final.label, p_final_platt, platt_t),
    "locked_late_fusion": metrics(final.label, final.probability_fusion, 0.5),
}
control_table = pd.DataFrame(control_results).T.reset_index(names="model")
control_table.to_csv(OUTPUT_ROOT / "fair_text_baseline_comparison.csv", index=False)
save_json("fair_text_baseline_comparison.json", {
    "validation_selected_text_threshold": best_t,
    "platt_calibration_fit_n": int(len(idx_cal)), "platt_threshold_tuning_n": int(len(idx_tune)),
    "platt_selected_threshold": platt_t, "final_results": control_results
})
display(control_table[["model","accuracy","macro_f1","roc_auc","pr_auc","brier","log_loss","threshold"]])

## 4. Source, domain, and time diagnostics

These are subgroup diagnostics, not a source-disjoint test: the trained models may have seen the same source categories during training. A source-only baseline is also fitted from original training labels. If it performs well, source identity alone carries target information and remains a plausible shortcut even though metadata was withheld from the neural networks.

In [ ]:
test_meta = read_raw("test", ["id","subreddit","domain","created_utc","2_way_label"])
test_meta["id"] = test_meta["id"].astype(str); final["id"] = final["id"].astype(str)
enriched = final.merge(test_meta, on="id", how="left", validate="one_to_one")
enriched["year"] = pd.to_datetime(enriched.created_utc, unit="s", errors="coerce", utc=True).dt.year

def subgroup_table(group, min_n=30):
    rows=[]
    for key, g in enriched.groupby(group, dropna=False):
        if len(g) < min_n or g.label.nunique() < 2: continue
        tm=metrics(g.label, g.probability_roberta); fm=metrics(g.label, g.probability_fusion)
        rows.append({group:key, "n":len(g), "class_1_rate":g.label.mean(),
                     "roberta_macro_f1":tm["macro_f1"], "fusion_macro_f1":fm["macro_f1"],
                     "fusion_delta_macro_f1":fm["macro_f1"]-tm["macro_f1"],
                     "roberta_accuracy":tm["accuracy"], "fusion_accuracy":fm["accuracy"]})
    columns=[group,"n","class_1_rate","roberta_macro_f1","fusion_macro_f1",
             "fusion_delta_macro_f1","roberta_accuracy","fusion_accuracy"]
    return pd.DataFrame(rows, columns=columns).sort_values("n", ascending=False)

for group in ["subreddit","domain","year"]:
    table=subgroup_table(group)
    table.to_csv(OUTPUT_ROOT / f"final_performance_by_{group}.csv", index=False)
    display(table.head(15))

train_source = read_raw("train", ["subreddit","2_way_label"])
global_rate = float(train_source["2_way_label"].mean())
rates = train_source.groupby("subreddit")["2_way_label"].agg(["mean","size"])
smooth = (rates["mean"]*rates["size"] + global_rate*50)/(rates["size"]+50)
source_prob = enriched["subreddit"].map(smooth).fillna(global_rate).to_numpy()
source_only = metrics(enriched.label, source_prob, 0.5)

source_label_tab = pd.crosstab(train_source.subreddit, train_source["2_way_label"])
chi2, p, _, _ = chi2_contingency(source_label_tab)
v = math.sqrt(chi2/(source_label_tab.to_numpy().sum()*min(source_label_tab.shape[0]-1, source_label_tab.shape[1]-1)))
source_summary={"source_only_final_holdout":source_only,"training_source_label_cramers_v":float(v),"chi_square_p":float(p)}
save_json("source_shortcut_audit.json", source_summary)
print(source_summary)

## 5. Perceptual duplicate screening

Exact hashes miss resized or recompressed copies. A 64-bit perceptual hash is computed for every usable development image and final image. A BK-tree searches for cross-set pairs within Hamming distance 6. These pairs are candidates for manual review; perceptual similarity alone is not proof of duplication.

In [ ]:
def resolve_original(path_value):
    p=Path(str(path_value)); return p if p.is_absolute() else ORIGINAL_ROOT/p

def phash64(path):
    with Image.open(path) as im:
        a=np.asarray(im.convert("L").resize((32,32), Image.Resampling.LANCZOS), dtype=float)
    low=dctn(a, type=2, norm="ortho")[:8,:8]
    med=np.median(low.ravel()[1:])
    bits=(low>med).ravel()
    value=0
    for bit in bits: value=(value<<1)|int(bit)
    return value

cache_path=OUTPUT_ROOT/"perceptual_hashes.csv"
if cache_path.exists():
    hashes=pd.read_csv(cache_path, dtype={"id":str,"phash_hex":str})
else:
    image_rows=[]
    for split in ["train","validation","test"]:
        d=pd.read_parquet(PROCESSED_ROOT/"development_clean"/f"{split}.parquet")
        image_rows.extend(("development",str(r.id),resolve_original(r.image_path)) for r in d.itertuples())
    hold=pd.read_parquet(PROCESSED_ROOT/"final_holdout"/"final_5000.parquet")
    image_rows.extend(("final",str(r.id),resolve_original(r.image_path)) for r in hold.itertuples())
    out=[]
    for i,(partition,sid,path) in enumerate(image_rows,1):
        try: out.append({"partition":partition,"id":sid,"path":str(path),"phash_hex":f"{phash64(path):016x}","error":""})
        except Exception as e: out.append({"partition":partition,"id":sid,"path":str(path),"phash_hex":"","error":type(e).__name__})
        if i%2500==0: print(f"Hashed {i:,}/{len(image_rows):,}")
    hashes=pd.DataFrame(out); hashes.to_csv(cache_path,index=False)

class BKNode:
    def __init__(self,key,payload): self.key,self.payload,self.children=key,[payload],{}
    def add(self,key,payload):
        node=self
        while True:
            dist=(node.key^key).bit_count()
            if dist==0: node.payload.append(payload); return
            if dist not in node.children: node.children[dist]=BKNode(key,payload); return
            node=node.children[dist]
    def search(self,key,radius):
        found=[]; stack=[self]
        while stack:
            node=stack.pop(); dist=(node.key^key).bit_count()
            if dist<=radius: found.extend((p,dist) for p in node.payload)
            stack.extend(child for edge,child in node.children.items() if dist-radius<=edge<=dist+radius)
        return found

dev=hashes[(hashes.partition=="development") & hashes.phash_hex.ne("")]
fin=hashes[(hashes.partition=="final") & hashes.phash_hex.ne("")]
tree=None
for r in dev.itertuples():
    key=int(r.phash_hex,16)
    if tree is None: tree=BKNode(key,(r.id,r.path,r.phash_hex))
    else: tree.add(key,(r.id,r.path,r.phash_hex))
pairs=[]
for r in fin.itertuples():
    for (dev_id,dev_path,dev_hash),distance in tree.search(int(r.phash_hex,16),6):
        pairs.append({"final_id":r.id,"development_id":dev_id,"hamming_distance":distance,
                      "final_path":r.path,"development_path":dev_path,"final_phash":r.phash_hex,"development_phash":dev_hash})
near=pd.DataFrame(pairs).sort_values(["hamming_distance","final_id"]) if pairs else pd.DataFrame(columns=["final_id","development_id","hamming_distance"])
near.to_csv(OUTPUT_ROOT/"perceptual_near_duplicate_candidates.csv",index=False)
print(f"Usable hashes: development={len(dev):,}; final={len(fin):,}")
print(f"Cross-set candidate pairs at Hamming distance <=6: {len(near):,}")
display(near.head(20))

## 6. Stronger split manifests for future retraining

The current checkpoints cannot be converted into source-disjoint or chronological experiments after training. This section creates reproducible manifests for those experiments without running expensive model training. Each manifest uses only the existing cleaned multimodal development examples.

In [ ]:
clean_parts=[]
for split in ["train","validation","test"]:
    d=pd.read_parquet(PROCESSED_ROOT/"development_clean"/f"{split}.parquet")
    meta=read_raw(split,["id","subreddit","domain","created_utc","2_way_label"])
    d["id"]=d.id.astype(str); meta["id"]=meta.id.astype(str)
    clean_parts.append(d.merge(meta,on="id",how="left",validate="one_to_one"))
clean=pd.concat(clean_parts,ignore_index=True).drop_duplicates("id")
clean["created_datetime"]=pd.to_datetime(clean.created_utc,unit="s",errors="coerce",utc=True)

# Source-disjoint: Fakeddit subreddits are label-pure. Allocate whole subreddits
# separately inside each label so every split contains both classes.
purity=clean.groupby("subreddit")["label"].nunique()
assert purity.max()==1, "Expected subreddit-level distant-supervision labels."
source_labels=clean.groupby("subreddit")["label"].first()
counts=clean.groupby("subreddit").size()
assignment={}
for label in sorted(clean.label.unique()):
    label_counts=counts[source_labels==label].sort_values(ascending=False)
    label_total=int(label_counts.sum())
    targets={"train":.70*label_total,"validation":.15*label_total,"test":.15*label_total}
    totals={k:0 for k in targets}
    for source,count in label_counts.items():
        split=min(targets,key=lambda k: totals[k]/targets[k])
        assignment[source]=split; totals[split]+=int(count)
source_manifest=clean[["id","label","text","image_path","subreddit","domain","created_utc"]].copy()
source_manifest["v2_split"]=source_manifest.subreddit.map(assignment)
assert source_manifest.groupby("subreddit").v2_split.nunique().max()==1
assert source_manifest.groupby("v2_split").label.nunique().min()==2
source_manifest.to_parquet(MANIFEST_ROOT/"source_disjoint.parquet",index=False)

chronological=clean.sort_values(["created_datetime","id"]).copy()
chronological["v2_split"]="train"
chronological.iloc[int(.70*len(chronological)):int(.85*len(chronological)), chronological.columns.get_loc("v2_split")]="validation"
chronological.iloc[int(.85*len(chronological)):, chronological.columns.get_loc("v2_split")]="test"
chronological[["id","label","text","image_path","subreddit","domain","created_utc","v2_split"]].to_parquet(
    MANIFEST_ROOT/"chronological.parquet",index=False)

split_summary=pd.concat([
    source_manifest.groupby("v2_split").agg(n=("id","size"),class_1_rate=("label","mean")).assign(manifest="source_disjoint"),
    chronological.groupby("v2_split").agg(n=("id","size"),class_1_rate=("label","mean")).assign(manifest="chronological")
]).reset_index()
split_summary.to_csv(OUTPUT_ROOT/"v2_manifest_summary.csv",index=False)
display(split_summary)

## 7. Explainability coverage and reporting checks

This verifies whether the stored artifacts match the method description and writes a machine-readable correction list for the dissertation draft.

In [ ]:
cases32=pd.read_csv(RESULTS_ROOT/"case_studies"/"all_32_multimodal_cases.csv")
cases8=pd.read_csv(RESULTS_ROOT/"case_studies"/"selected_8_case_studies.csv")
shap_summary=pd.read_csv(RESULTS_ROOT/"shap"/"shap_case_summary.csv")
grad_summary=pd.read_csv(RESULTS_ROOT/"gradcam"/"gradcam_case_summary.csv")
coverage={
    "all_multimodal_cases":int(len(cases32)), "selected_paired_cases":int(len(cases8)),
    "shap_cases":int(len(shap_summary)), "gradcam_cases":int(len(grad_summary)),
    "case_columns":list(cases32.columns),
}
if "label" in cases32.columns: coverage["all_32_label_counts"]={str(k):int(v) for k,v in cases32.label.value_counts().items()}
save_json("explainability_coverage.json",coverage)

raw_test_n=len(read_raw("test",["id"]))
robustness=json.loads((RESULTS_ROOT/"metrics"/"missing_modality_robustness"/"results.json").read_text())
changes={r["scenario"]:r["macro_f1_change_vs_full"] for r in robustness["results"]}
checks=pd.DataFrame([
    {"item":"public test size","observed":raw_test_n,"expected":59319,"status":"PASS" if raw_test_n==59319 else "FAIL","draft_action":"Use 59,319 consistently."},
    {"item":"missing text change sign","observed":changes.get("missing_text_image_fallback"),"expected":"negative","status":"PASS" if changes.get("missing_text_image_fallback",1)<0 else "FAIL","draft_action":"Report -0.0906, not +0.0906."},
    {"item":"32-case table exists","observed":len(cases32),"expected":32,"status":"PASS" if len(cases32)==32 else "FAIL","draft_action":"Describe the full 32-case analysis and distinguish it from the four illustrated cases."},
    {"item":"paired SHAP and Grad-CAM coverage","observed":f"SHAP={len(shap_summary)}, Grad-CAM={len(grad_summary)}","expected":">=32 each","status":"PASS" if min(len(shap_summary),len(grad_summary))>=32 else "FAIL","draft_action":"Reconcile claimed and generated explanation counts."},
])
checks.to_csv(OUTPUT_ROOT/"reporting_consistency_checks.csv",index=False)
display(checks)
print(coverage)

## Interpretation boundaries

The strongest supported conclusion remains specific: on the selected cleaned Fakeddit sample and locked checkpoints, late probability fusion improves over the original RoBERTa decision rule. The added controls show whether that advantage remains after fair threshold tuning and how strongly source and selection effects constrain generalisation.

The source-disjoint and chronological manifests must be used to retrain all models from scratch before making claims about transfer to unseen sources or later periods. Full training-seed uncertainty likewise requires several independent end-to-end training runs; it cannot be recovered from one checkpoint. This notebook deliberately does not fabricate those results.